# 08.2 — Generation Metrics: BLEU, ROUGE, BERTScore

**The fundamental problem:** How do you automatically measure the quality of generated text? Unlike classification, there's rarely a single correct answer — "The cat sat" and "A feline was seated" can both be perfect translations.

**Three families:**
- **BLEU**: n-gram precision (generation quality). Machine translation.
- **ROUGE**: n-gram recall (coverage). Summarization.
- **BERTScore**: semantic similarity (embedding-based). Handles paraphrase.

---

In [ ]:
import re
import math
import numpy as np
from collections import Counter
from typing import List

def tokenize(text: str) -> List[str]:
    return re.findall(r'\b[a-z]+\b', text.lower())

def ngrams(tokens: List[str], n: int) -> List[tuple]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


# ---- BLEU (Bilingual Evaluation Understudy) ----
# Papineni et al., 2002. Standard metric for machine translation.
#
# BLEU = BP × exp(sum_n w_n × log(p_n))
# where:
# - p_n = clipped n-gram precision for n=1..N
# - BP = brevity penalty (penalizes short outputs)
# - w_n = 1/N (uniform weights)

def clipped_ngram_precision(hypothesis: List[str], references: List[List[str]], n: int) -> float:
    """
    Clipped precision: for each n-gram in hypothesis,
    count = min(count_in_hyp, max_count_in_any_reference).
    
    'Clipping' prevents gaming by repeating common words.
    Without clipping: 'the the the the' would have precision 1.0 if 'the' is in reference.
    """
    hyp_ngrams = Counter(ngrams(hypothesis, n))
    if not hyp_ngrams:
        return 0.0
    
    # For each n-gram in hypothesis, max count across references
    max_ref_counts = Counter()
    for ref in references:
        ref_ngrams = Counter(ngrams(ref, n))
        for ng, count in ref_ngrams.items():
            max_ref_counts[ng] = max(max_ref_counts[ng], count)
    
    # Clipped count
    clipped = sum(min(count, max_ref_counts[ng]) for ng, count in hyp_ngrams.items())
    total = sum(hyp_ngrams.values())
    return clipped / total


def brevity_penalty(hypothesis: List[str], references: List[List[str]]) -> float:
    """
    BP = 1 if len(hyp) > len(ref), else exp(1 - len(ref)/len(hyp)).
    Penalizes hypothesis that is shorter than reference.
    Without BP: generate 1 word with high precision → perfect score.
    """
    c = len(hypothesis)
    # Use the reference with length closest to hypothesis
    r = min(len(ref) for ref in references, key=lambda x: abs(x - c))
    if c >= r:
        return 1.0
    return math.exp(1 - r / c) if c > 0 else 0.0


def bleu_score(hypothesis: str, references: List[str], max_n=4) -> dict:
    """
    Compute BLEU-N score.
    Returns individual n-gram scores and the overall BLEU.
    """
    hyp_tokens = tokenize(hypothesis)
    ref_tokens_list = [tokenize(r) for r in references]
    
    # Compute clipped precision for each n
    precisions = []
    for n in range(1, max_n + 1):
        p = clipped_ngram_precision(hyp_tokens, ref_tokens_list, n)
        precisions.append(p)
    
    bp = brevity_penalty(hyp_tokens, ref_tokens_list)
    
    # Geometric mean of precisions (log-space for stability)
    if any(p == 0 for p in precisions):
        bleu = 0.0
    else:
        log_avg = sum(math.log(p) for p in precisions) / max_n
        bleu = bp * math.exp(log_avg)
    
    return {
        'bleu': bleu,
        'bp': bp,
        **{f'p{n}': precisions[n-1] for n in range(1, max_n+1)}
    }


# Test BLEU
ref = "The cat sat on the mat"
hyp_perfect = "The cat sat on the mat"
hyp_good    = "The cat is sitting on a mat"
hyp_short   = "cat sat mat"
hyp_repeat  = "the the the the the the"

print("BLEU scores:")
for name, hyp in [("Perfect", hyp_perfect), ("Good", hyp_good),
                   ("Short", hyp_short), ("Repeated", hyp_repeat)]:
    scores = bleu_score(hyp, [ref])
    print(f"  {name:10}: BLEU={scores['bleu']:.4f}  BP={scores['bp']:.3f}  "
          f"P1={scores['p1']:.3f} P2={scores['p2']:.3f} P3={scores['p3']:.3f} P4={scores['p4']:.3f}")

In [ ]:
# ---- ROUGE (Recall-Oriented Understudy for Gisting Evaluation) ----
# Lin, 2004. Standard metric for summarization.
# Unlike BLEU (precision-focused), ROUGE measures recall:
# how much of the reference is covered by the hypothesis?

def rouge_n(hypothesis: str, reference: str, n: int) -> dict:
    """
    ROUGE-N: n-gram recall between hypothesis and reference.
    Also computes precision and F1 for a complete picture.
    """
    hyp_tok = tokenize(hypothesis)
    ref_tok = tokenize(reference)
    
    hyp_ngrams = Counter(ngrams(hyp_tok, n))
    ref_ngrams = Counter(ngrams(ref_tok, n))
    
    # Overlap
    overlap = sum((hyp_ngrams & ref_ngrams).values())
    
    precision = overlap / sum(hyp_ngrams.values()) if hyp_ngrams else 0.0
    recall    = overlap / sum(ref_ngrams.values()) if ref_ngrams else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {'precision': precision, 'recall': recall, 'f1': f1}


def lcs_length(a: List[str], b: List[str]) -> int:
    """
    Longest Common Subsequence (not substring — gaps allowed).
    Used in ROUGE-L.
    """
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]


def rouge_l(hypothesis: str, reference: str) -> dict:
    """
    ROUGE-L: LCS-based recall. Captures long-range order.
    Better than ROUGE-N for sentences where word order matters
    but exact contiguous n-grams aren't required.
    """
    hyp_tok = tokenize(hypothesis)
    ref_tok = tokenize(reference)
    
    lcs = lcs_length(hyp_tok, ref_tok)
    precision = lcs / len(hyp_tok) if hyp_tok else 0.0
    recall    = lcs / len(ref_tok) if ref_tok else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'lcs': lcs}


# Test ROUGE on summarization example
reference_summary = (
    "The transformer model uses self-attention mechanisms to process sequences in parallel. "
    "It was introduced in 2017 and has become the foundation of modern NLP."
)
generated_summary = (
    "Transformers use attention to process tokens simultaneously, "
    "forming the basis of NLP since their introduction in 2017."
)
extractive_summary = (
    "The transformer model uses self-attention mechanisms to process sequences. "
    "It has become the foundation of modern NLP."
)

print("ROUGE scores (vs reference):")
print(f"{'':20} ROUGE-1 F1  ROUGE-2 F1  ROUGE-L F1")
for name, summ in [("Generated", generated_summary), ("Extractive", extractive_summary)]:
    r1 = rouge_n(summ, reference_summary, 1)['f1']
    r2 = rouge_n(summ, reference_summary, 2)['f1']
    rl = rouge_l(summ, reference_summary)['f1']
    print(f"  {name:18}: {r1:.4f}       {r2:.4f}       {rl:.4f}")

print()
print("Extractive summary usually scores higher (copies words from document).")
print("Abstractive summaries can be better quality but score lower on n-gram metrics.")
print("This is ROUGE's main weakness — it punishes paraphrase.")

In [ ]:
# ---- BERTScore ----
# Zhang et al., 2019.
# Instead of exact n-gram overlap, compute cosine similarity between
# contextual embeddings from BERT.
#
# For each token in hypothesis, find the most similar token in reference.
# Precision: max over reference for each hypothesis token
# Recall: max over hypothesis for each reference token

def bertscore_from_embeddings(hyp_embs: np.ndarray, ref_embs: np.ndarray,
                               idf_weights: np.ndarray = None) -> dict:
    """
    Compute BERTScore given pre-computed token embeddings.
    
    hyp_embs: [T_h, D] normalized embeddings of hypothesis tokens
    ref_embs: [T_r, D] normalized embeddings of reference tokens
    idf_weights: optional per-token IDF weights (down-weight common tokens)
    """
    # Pairwise cosine similarity: [T_h, T_r]
    sim_matrix = hyp_embs @ ref_embs.T  # normalized -> dot = cosine
    
    # Precision: for each hyp token, best match in reference
    max_sim_hyp = sim_matrix.max(axis=1)  # [T_h]
    # Recall: for each ref token, best match in hypothesis
    max_sim_ref = sim_matrix.max(axis=0)  # [T_r]
    
    if idf_weights is not None:
        # IDF-weighted (not implemented here for simplicity)
        pass
    
    P = max_sim_hyp.mean()
    R = max_sim_ref.mean()
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    return {'precision': float(P), 'recall': float(R), 'f1': float(F1)}


# Simulate token embeddings (in practice: run BERT and extract layer 9 reps)
def simulate_embeddings(tokens: List[str], d: int = 64) -> np.ndarray:
    """
    Simulate BERT embeddings. Real BERTScore uses a specific BERT layer.
    Paraphrases should have similar embeddings.
    """
    np.random.seed(42)
    # Use a fixed embedding per word type (ignores context, unlike real BERT)
    word_embs = {}
    for t in set(tokens):
        if t not in word_embs:
            word_embs[t] = np.random.randn(d)
    embs = np.array([word_embs[t] for t in tokens])
    # L2 normalize
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    return embs / np.maximum(norms, 1e-9)


ref_tokens = tokenize(reference_summary)
hyp_tokens = tokenize(generated_summary)
ext_tokens = tokenize(extractive_summary)

# Combine vocab for consistent embeddings
all_tokens = ref_tokens + hyp_tokens + ext_tokens
_ = simulate_embeddings(all_tokens)  # seed consistently

ref_embs = simulate_embeddings(ref_tokens)
hyp_embs = simulate_embeddings(hyp_tokens)
ext_embs = simulate_embeddings(ext_tokens)

print("BERTScore (simulated embeddings):")
for name, embs in [("Generated", hyp_embs), ("Extractive", ext_embs)]:
    scores = bertscore_from_embeddings(embs, ref_embs)
    print(f"  {name:12}: P={scores['precision']:.4f}  R={scores['recall']:.4f}  F1={scores['f1']:.4f}")

print()
print("Real BERTScore uses contextual embeddings from BERT layer 9 (typically).")
print("Paraphrases get high scores because similar meaning = similar embedding.")
print("This is the key advantage over BLEU/ROUGE.")

In [ ]:
# ---- Metric comparison: when to use what ----

print("=== Generation Metric Comparison ===")
print()

metrics = [
    ("BLEU",
     "MT, structured generation",
     "Doesn't penalize extra words (precision only), no paraphrase support",
     "Standard in MT papers; use BLEU-4 for sentences"),
    
    ("ROUGE-1/2",
     "Summarization (word overlap)",
     "Precision/recall tradeoff depends on output length",
     "ROUGE-2 F1 most common in summarization"),
    
    ("ROUGE-L",
     "Summarization (order matters)",
     "Slightly better than ROUGE-2 when sentence order matters",
     "Use alongside ROUGE-2"),
    
    ("BERTScore",
     "Any generation with paraphrase tolerance",
     "Slow (needs BERT), harder to interpret",
     "Higher correlation with human judgments than BLEU/ROUGE"),
    
    ("CIDEr",
     "Image captioning",
     "Requires multiple references, specialized to captioning",
     "TF-IDF weighted consensus"),
    
    ("METEOR",
     "MT (paraphrase-aware)",
     "Complex, slower than BLEU",
     "Uses synonym matching; higher human correlation than BLEU"),
]

for name, use_case, weakness, note in metrics:
    print(f"  {name:12}: {use_case}")
    print(f"  {'':12}  - Weakness: {weakness}")
    print(f"  {'':12}  Note: {note}")
    print()

print("=== The dirty secret ===")
print()
print("All automatic metrics correlate weakly with human judgments.")
print("BLEU-4 has Pearson r ≈ 0.5-0.7 with humans on MT tasks.")
print("BERTScore: ~0.6-0.8.")
print()
print("For production systems: always run human evaluation.")
print("Use automatic metrics for rapid development iteration only.")
print("See 08.3: LLM-as-judge for a more scalable quality signal.")

## Summary

**BLEU:** Clipped n-gram precision × brevity penalty. Designed for MT. Punishes short outputs. Doesn't reward paraphrase.

**ROUGE:** N-gram recall (and F1). Designed for summarization. Higher recall = more reference content covered.

**BERTScore:** Token-level cosine similarity in BERT embedding space. Handles synonyms and paraphrases. Most correlated with human judgment.

**Practical takeaway:** Report ROUGE-2 F1 + BERTScore F1 together. ROUGE measures surface overlap, BERTScore measures semantic overlap. Disagreement between them is informative.

---
**Next:** 08.3 — LLM as Judge